Fuente:
Datos abiertos  de SEGSOCIAL	IMSS-BIENESTAR	S038 / S038	10,773,410 registros JULIO - SEPTIEMBRE 2025
https://pub.bienestar.gob.mx/data/v2/last/S038/257_259/Programa_S038_S038_257_259.zip

In [24]:
from pathlib import Path

PROJECT_PATH = Path("/Users/daragama/Documents/Master25/Tesis/datos_semisinteticos/")
DATAPATH = PROJECT_PATH  / "data"
print(f"DATAPATH: {DATAPATH}")

DATAPATH: /Users/daragama/Documents/Master25/Tesis/datos_semisinteticos/data


In [48]:
# read all csv files in the data folder
import pandas as pd


raw_datapath = DATAPATH / "raw" / "datos_bienestar"
csv_files = list(raw_datapath.glob("*.csv"))

# crear un df con todos los csv files
df_nombres = pd.concat([pd.read_csv(f, encoding="latin-1", low_memory=False) for f in csv_files], ignore_index=True)

# df_nombres = pd.read_csv(csv_file, encoding="latin-1", low_memory=False) # check encoding if needed
df_nombres.head()

,NUM.,CVE_ENTIDAD,CVE_MUNICIPIO,PRIMER APELLIDO,SEGUNDO APELLIDO,NOMBRE,SEXO,EDAD,FECHA ALTA,IMPORTE BENEFICIO
0,5046.0,24.0,24032.0,MENOR,MENOR,MENOR,MENOR,MENOR,05-05-2015,0.0
1,5047.0,24.0,24032.0,ROMERO,MARTINEZ,JOSE GUADALUPE,MASCULINO,34,05-05-2015,0.0
2,5048.0,24.0,24032.0,ROMERO,MENDOZA,MANUELA,FEMENINO,65,05-05-2015,0.0
3,5049.0,24.0,24032.0,PADRON,SANCHEZ,JUAN,MASCULINO,94,05-05-2015,0.0
4,5050.0,24.0,24032.0,AGUILAR,CARDENAS,BENITA,FEMENINO,63,05-05-2015,0.0


In [49]:
df_nombres.info()

<class 'pandas.DataFrame'>
RangeIndex: 10773422 entries, 0 to 10773421
Data columns (total 10 columns):
 #   Column             Dtype  
---  ------             -----  
 0   NUM.               str    
 1   CVE_ENTIDAD        float64
 2   CVE_MUNICIPIO      float64
 3   PRIMER APELLIDO    str    
 4   SEGUNDO APELLIDO   str    
 5   NOMBRE             str    
 6   SEXO               str    
 7   EDAD               str    
 8   FECHA ALTA         str    
 9   IMPORTE BENEFICIO  float64
dtypes: float64(3), str(7)
memory usage: 821.9 MB


In [50]:
# buscar registros con CVE_ENTIDAD, NOMBRE, PRIMER APELLIDO, SEGUNDO APELLIDO y EDAD vacíos
empties = df_nombres[df_nombres[["CVE_ENTIDAD", "NOMBRE", "PRIMER APELLIDO", "SEGUNDO APELLIDO", "EDAD"]].isnull().all(axis=1)]
# eliminar estos registros
df_nombres = df_nombres.drop(empties.index)

df_nombres.isnull().sum()

NUM.                   0
CVE_ENTIDAD            0
CVE_MUNICIPIO          0
PRIMER APELLIDO      441
SEGUNDO APELLIDO      20
NOMBRE                 0
SEXO                   0
EDAD                   0
FECHA ALTA             0
IMPORTE BENEFICIO      0
dtype: int64

In [51]:
# menores tiene MENOR en todos los campos
df_nombres = df_nombres[df_nombres["EDAD"] != "MENOR"]

# EDAD

In [52]:
df_nombres.value_counts("EDAD")

EDAD
18     235851
19     230066
20     229151
21     226459
22     223560
        ...  
96       4344
97       3538
98       2924
99       2739
100         5
Name: count, Length: 84, dtype: int64

In [53]:
bins = [14, 19, 24, 29, 34, 39, 44, 49, 54, 59, 64,
        69, 74, 79, 84, 89, 94, 99, 104, 109, 114, 119, float("inf")]

labels = [
    "De 15 a 19", "De 20 a 24", "De 25 a 29", "De 30 a 34",
    "De 35 a 39", "De 40 a 44", "De 45 a 49", "De 50 a 54",
    "De 55 a 59", "De 60 a 64", "De 65 a 69", "De 70 a 74",
    "De 75 a 79", "De 80 a 84", "De 85 a 89", "De 90 a 94",
    "De 95 a 99", "De 100 a 104", "De 105 a 109", "De 110 a 114",
    "De 115 a 119", "De 120 y más"
]

edad_num = pd.to_numeric(df_nombres["EDAD"], errors="coerce")

df_nombres["EDAD_AGRUPADA"] = pd.cut(
    edad_num,
    bins=bins,
    labels=labels,
    right=True
)

df_nombres.head()

,NUM.,CVE_ENTIDAD,CVE_MUNICIPIO,PRIMER APELLIDO,SEGUNDO APELLIDO,NOMBRE,SEXO,EDAD,FECHA ALTA,IMPORTE BENEFICIO,EDAD_AGRUPADA
1,5047.0,24.0,24032.0,ROMERO,MARTINEZ,JOSE GUADALUPE,MASCULINO,34,05-05-2015,0.0,De 30 a 34
2,5048.0,24.0,24032.0,ROMERO,MENDOZA,MANUELA,FEMENINO,65,05-05-2015,0.0,De 65 a 69
3,5049.0,24.0,24032.0,PADRON,SANCHEZ,JUAN,MASCULINO,94,05-05-2015,0.0,De 90 a 94
4,5050.0,24.0,24032.0,AGUILAR,CARDENAS,BENITA,FEMENINO,63,05-05-2015,0.0,De 60 a 64
5,5051.0,24.0,24032.0,BORJAS,MARTINEZ,GABRIELA,FEMENINO,60,05-05-2015,0.0,De 60 a 64


# NOMBRES

In [54]:
# vienen ambos nombres en la columna NOMBRE, separar en NOMBRE y SEGUNDO NOMBRE
df_nombres[["NOMBRE", "SEGUNDO_NOMBRE"]] = df_nombres["NOMBRE"].str.split(" ", n=1, expand=True)

In [55]:
# unique nombres por edad agrupada
df_agrupado_nombre = (
    df_nombres
    .groupby(["EDAD_AGRUPADA", "CVE_ENTIDAD", "NOMBRE"])
    .size()
    .reset_index(name="CONTEO")
)

df_agrupado_nombre.head(20)

,EDAD_AGRUPADA,CVE_ENTIDAD,NOMBRE,CONTEO
0,De 15 a 19,2.0,AARON,8
1,De 15 a 19,2.0,AARONANTONIO,1
2,De 15 a 19,2.0,ABDIEL,3
3,De 15 a 19,2.0,ABEL,3
4,De 15 a 19,2.0,ABELARDO,1
5,De 15 a 19,2.0,ABELRUBEN,1
6,De 15 a 19,2.0,ABIGAIL,3
7,De 15 a 19,2.0,ABIMAEL,1
8,De 15 a 19,2.0,ABISAI,1
9,De 15 a 19,2.0,ABRAAM,1


## ejemplo

In [56]:
edad_agrupada = "De 15 a 19"
entidad = 10

filtro_nombre = df_agrupado_nombre[
    (df_agrupado_nombre["EDAD_AGRUPADA"] == edad_agrupada) &
    (df_agrupado_nombre["CVE_ENTIDAD"] == entidad)
]

print("Máximo:", filtro_nombre["CONTEO"].max())
print("Mínimo:", filtro_nombre["CONTEO"].min())
print("Nombres únicos:", len(filtro_nombre["NOMBRE"].unique()))

Máximo: 425
Mínimo: 1
Nombres únicos: 2493


# SELECCIÓN

Probabilidades "media"

In [77]:
df_agrupado_nombre = (
    df_nombres
    .groupby([
        "EDAD_AGRUPADA",
        "CVE_ENTIDAD",
        "NOMBRE",
        # "SEGUNDO_NOMBRE"
    ], dropna=False)
    .size()
    .reset_index(name="CONTEO")
)

df_agrupado_nombre.head(20)

,EDAD_AGRUPADA,CVE_ENTIDAD,NOMBRE,CONTEO
0,De 15 a 19,2.0,AARON,8
1,De 15 a 19,2.0,AARONANTONIO,1
2,De 15 a 19,2.0,ABDIEL,3
3,De 15 a 19,2.0,ABEL,3
4,De 15 a 19,2.0,ABELARDO,1
5,De 15 a 19,2.0,ABELRUBEN,1
6,De 15 a 19,2.0,ABIGAIL,3
7,De 15 a 19,2.0,ABIMAEL,1
8,De 15 a 19,2.0,ABISAI,1
9,De 15 a 19,2.0,ABRAAM,1


In [78]:
import numpy as np

def seleccionar_nombre(edad_agrupada, entidad, frecuencia, df_agrupado):
    
    filtro = df_agrupado[
        (df_agrupado["EDAD_AGRUPADA"] == edad_agrupada) &
        (df_agrupado["CVE_ENTIDAD"] == entidad)
    ].copy()

    if filtro.empty:
        return None

    q33 = filtro["CONTEO"].quantile(1/3)
    q66 = filtro["CONTEO"].quantile(2/3)
    print("q33:", q33)
    print("q66:", q66)

    if q33 == q66:
        return filtro.sample(n=1)["NOMBRE"].iloc[0]  # selección aleatoria

    if frecuencia == "baja":
        nombres = filtro[filtro["CONTEO"] <= q33]

    elif frecuencia == "media":
        nombres = filtro[
            (filtro["CONTEO"] > q33) &
            (filtro["CONTEO"] <= q66)
        ]

    elif frecuencia == "alta":
        nombres = filtro[filtro["CONTEO"] > q66]

    else:
        raise ValueError("frecuencia debe ser 'alta', 'media' o 'baja'")

    if nombres.empty:
        return None # TODO, raise an exception or return a default value?????

    return nombres.sample(n=1)["NOMBRE"].iloc[0] # selección aleatoria

# no hay repetidos
edad_agrupada = "De 25 a 29"
entidad = 5



print(f"Nombre con alta frecuencia {seleccionar_nombre(edad_agrupada, entidad, "alta", df_agrupado_nombre)}")
print(f"Nombre con media frecuencia {seleccionar_nombre(edad_agrupada, entidad, "media", df_agrupado_nombre)}")
print(f"Nombre con baja frecuencia {seleccionar_nombre(edad_agrupada, entidad, "baja", df_agrupado_nombre)}")

q33: 1.0
q66: 2.0
Nombre con alta frecuencia CYNTHIA
q33: 1.0
q66: 2.0
Nombre con media frecuencia YOBANA
q33: 1.0
q66: 2.0
Nombre con baja frecuencia MABEL


> Si hay "bolsas" (edad + entidad) vacías